# Compare ALE / Coordinate / DiFuMo Models

Loads model-scoped comparison rows from Google Drive by default, with a local fallback. The table prioritizes the semantic `main_comparison_summary_row` artifacts and keeps exact PMID retrieval as a diagnostic.


## Setup

In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt

IN_COLAB = Path('/content').exists()
if IN_COLAB:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
    except Exception as exc:
        print('Drive mount skipped:', exc)

DRIVE_ROOT = Path('/content/drive/MyDrive/neurovlm_ale3dcnn')
LOCAL_ROOT = Path('.')
ROOT = DRIVE_ROOT if DRIVE_ROOT.exists() else LOCAL_ROOT
RUNS_DIR = ROOT / 'runs_ale_3dcnn'
COMPARISON_FILE = RUNS_DIR / 'ale_model_comparison.csv'
BASELINE_SUMMARY_CANDIDATES = [
    Path('/content/drive/MyDrive/neurovlm_pubmed_semantic_baseline/main_comparison_summary_row.json'),
    Path('experiments/runs/neurovlm_pubmed_semantic_baseline/main_comparison_summary_row.json'),
]
print('Using root:', ROOT)
print('Runs directory:', RUNS_DIR)
print('Comparison CSV:', COMPARISON_FILE)


## Load Results

In [ ]:
def load_json(path):
    try:
        return json.loads(Path(path).read_text())
    except Exception:
        return {}

def normalize_row(row):
    row = dict(row)
    if 'model_name' not in row and 'model' in row:
        row['model_name'] = row['model']
    if 'model' not in row and 'model_name' in row:
        row['model'] = row['model_name']
    if 'exact_pmid_paper_recall_curve_auc' not in row and 'paper_recall_curve_auc' in row:
        row['exact_pmid_paper_recall_curve_auc'] = row['paper_recall_curve_auc']
    if 'exact_pmid_recall@1' not in row and 'recall@1' in row:
        row['exact_pmid_recall@1'] = row['recall@1']
    if 'exact_pmid_recall@5' not in row and 'recall@5' in row:
        row['exact_pmid_recall@5'] = row['recall@5']
    if 'exact_pmid_recall@10' not in row and 'recall@10' in row:
        row['exact_pmid_recall@10'] = row['recall@10']
    if 'exact_pmid_recall@50' not in row and 'recall@50' in row:
        row['exact_pmid_recall@50'] = row['recall@50']
    if 'exact_pmid_mrr' not in row and 'mrr' in row:
        row['exact_pmid_mrr'] = row['mrr']
    if 'exact_pmid_median_rank' not in row and 'median_rank' in row:
        row['exact_pmid_median_rank'] = row['median_rank']
    return row

def load_comparison_rows(runs_dir, comparison_file):
    rows = []
    run_dirs = sorted({p.parent for p in runs_dir.glob('**/main_comparison_summary_row.json')})
    run_dirs.extend(p.parent for p in runs_dir.glob('**/comparison_row.json') if p.parent not in run_dirs)
    for run_dir in run_dirs:
        row = load_json(run_dir / 'comparison_row.json')
        sem = load_json(run_dir / 'main_comparison_summary_row.json')
        row.update(sem)
        row['_source_dir'] = str(run_dir)
        rows.append(normalize_row(row))
    for baseline_path in BASELINE_SUMMARY_CANDIDATES:
        if baseline_path.exists():
            row = normalize_row(load_json(baseline_path))
            row['_source_dir'] = str(baseline_path.parent)
            rows.append(row)
            break
    if rows:
        print(f'Loaded {len(rows)} run/baseline summary rows from {runs_dir}')
        return pd.DataFrame(rows)
    if comparison_file.exists():
        print(f'Loading aggregate CSV: {comparison_file}')
        return pd.read_csv(comparison_file).apply(lambda r: normalize_row(r.to_dict()), axis=1, result_type='expand')
    raise FileNotFoundError(
        f'No semantic summaries, comparison rows, or aggregate CSV found under {runs_dir}. '
        'Run the training notebook first, or set DRIVE_ROOT/RUNS_DIR to the folder containing runs.'
    )

df = load_comparison_rows(RUNS_DIR, COMPARISON_FILE)
df.tail()


## Optional Manual Baselines

In [ ]:
# Optional rows you can paste in from other model families if they are not in this Drive tree.
MANUAL_BASELINES = []
if MANUAL_BASELINES:
    df = pd.concat([df, pd.DataFrame(MANUAL_BASELINES).apply(lambda r: normalize_row(r.to_dict()), axis=1, result_type='expand')], ignore_index=True)
df


## Main Table

In [ ]:
metric_col = 'mesh_recall@10' if 'mesh_recall@10' in df.columns else 'exact_pmid_paper_recall_curve_auc'
cols = [
    'model', 'model_name', 'preprocessing_type', 'graph_type', 'uses_difumo', 'atlas_free',
    'n_test_pmids', 'params', 'number_of_parameters', 'peak_vram', 'training_time_per_epoch',
    'exact_pmid_paper_recall_curve_auc', 'exact_pmid_recall@1', 'exact_pmid_recall@5',
    'exact_pmid_recall@10', 'exact_pmid_recall@50', 'exact_pmid_mrr', 'exact_pmid_median_rank',
    'network_accuracy', 'network_top_2_accuracy', 'network_macro_auc',
    'mesh_recall@5', 'mesh_recall@10', 'mesh_map', 'mesh_mrr', 'mesh_median_best_true_term_rank',
    'semantic_recall@10', 'semantic_recall@50', 'semantic_mrr', 'semantic_paper_style_recall_curve_auc',
    'checkpoint_path', 'config_path', '_source_dir',
]
available_cols = [c for c in cols if c in df.columns]
table = df[available_cols].copy()
table = table.sort_values(metric_col, ascending=False, na_position='last')
table


## Semantic Metric Plot


In [ ]:
plot_df = table.copy()
label_col = 'model' if 'model' in plot_df.columns else 'model_name'
plot_cols = [c for c in ['network_accuracy', 'mesh_recall@10', 'semantic_recall@10', 'exact_pmid_recall@10'] if c in plot_df.columns]
fig, ax = plt.subplots(figsize=(10, max(4, 0.45 * len(plot_df))))
if plot_cols and len(plot_df):
    plot_df.set_index(label_col)[plot_cols].plot(kind='barh', ax=ax)
    ax.set_xlabel('score')
    ax.set_title('Semantic Metrics and Exact PMID Diagnostic')
    ax.invert_yaxis()
    ax.legend(loc='best')
plt.tight_layout()


## Exact PMID Diagnostic Plot


In [ ]:
exact_cols = [c for c in ['exact_pmid_recall@1', 'exact_pmid_recall@5', 'exact_pmid_recall@10', 'exact_pmid_recall@50'] if c in table.columns]
fig, ax = plt.subplots(figsize=(9, 4.5))
if exact_cols and len(table):
    labels = table[label_col].astype(str)
    for col in exact_cols:
        ax.plot(labels, table[col], marker='o', label=col)
    ax.tick_params(axis='x', rotation=45)
    ax.set_ylabel('recall')
    ax.set_title('Exact PMID Retrieval Diagnostic')
    ax.legend()
plt.tight_layout()


## Interpretation Template

- Exact PMID retrieval is a strict diagnostic, not the main success metric.
- CNN/GNN/DiFuMo variants should only be treated as better than the pretrained NeuroVLM MLP if they improve one or more semantic metrics on the same official PubMed test split.
- Prioritize network labeling, MeSH term ranking, and semantic-neighborhood retrieval when judging semantic usefulness.
- If exact PMID recall does not improve but semantic metrics do, that is still evidence the model is useful for interpretation.
